# 41. The second membership gate: six new candidates

**One variable against ledger row 77** (`stack_35_all5`, CV 0.968110, public LB 0.96941): the
member set. Same logistic combiner, same `C=1.0`, same logit meta-features, same fold-wise
protocol, same folds. Row 77 is refit here as the reproduction check and the paired baseline.

Six candidates, all produced 2026-08-22 and logged as rows 83 to 93:

| candidate | solo CV | what it is |
|---|---|---|
| `xgb_te_fe` | **0.968005** | row 93, the best single model in the repo |
| `cat_te_n4000` | 0.967166 | row 87, CatBoost at its fitted budget |
| `xgb_raw_fe` | 0.964666 | row 91, no encoder, ratio block on |
| `cat_raw_n10000` | 0.964042 | row 83, no encoder, at its fitted budget |
| `lgb_raw_fe` | 0.963821 | row 90, no encoder, ratio block on |
| `cat_raw_fe` | 0.962181 | row 92, no encoder, ratio block on |

**Three of the six are improved versions of members already in the stack**, not new families:
`xgb_te_fe` improves on `xgb_te`, `cat_te_n4000` on `cat42`, `cat_raw_n10000` on `cat_raw`. The
originals are deliberately KEPT rather than replaced. Row 77 is the reason: `cat_native_c1` and
`cat_native_c2` entered as a near-antisymmetric pair at -0.3007 and +0.2939, so what the combiner
extracted was their *difference*. A model and its improved twin are exactly that shape, and
swapping rather than adding would throw the difference away. If the twins turn out to be
redundant the combiner will price them near zero, which is a cheaper way to find out than
pre-judging it.

**None of the six is a duplicate.** `cat_te_n2000` and `cat_raw_n2000` ARE bit-identical to
members already present, which is why neither is in the list above; the hash quarantine below
would have caught them anyway and would have failed the run.

## The pre-registered gate

Unchanged since rows 26 and 34: **positive on at least 4 of 5 folds and at least +0.00005 on the
paired mean.** Gate, membership and submission stay three separate decisions.

## What row 77 taught, and how this notebook applies it

Row 77's finding was that **no single one of its five candidates cleared the floor while the set
cleared it by 2.9x the sum of the parts.** Testing one at a time would have rejected all five.
That was a hole in every gate this repo ran before it.

So the all-six arm is the **primary** arm here and the singles are diagnostic. A single arm coming
in under the floor is not evidence against the set, and this notebook will not read it as such.

## The prediction, written before the run

**The set clears the floor.** Beyond that I am deliberately not predicting a magnitude. My last
two magnitude predictions were wrong in opposite directions, by fourfold in row 39 and backwards
in row 40, and both were mechanism arguments invented for the occasion. The one thing with a
track record here is row 77's superadditivity result, so the only claim worth making is the one
it directly supports: the set beats the best single arm.

The honest case against: three of six are twins of existing members, and row 28 pruned five
members that were carrying nothing. A stack of 41 with three near-duplicate pairs may simply be
absorbing them at zero.

In [1]:
# 37_stack_views.ipynb
# Membership gate for the five vectors produced by notebooks 35 and 36.
# Runs locally: every member vector already lives in artifacts/oof.
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(b for b in [Path.cwd(), *Path.cwd().parents]
            if (b / "data" / "raw" / "train.csv").exists())
O = ROOT / "artifacts" / "oof"
S = ROOT / "submissions"

train = pd.read_csv(ROOT / "data" / "raw" / "train.csv")
test = pd.read_csv(ROOT / "data" / "raw" / "test.csv")
y = train["addicted_label"].to_numpy(np.int8)

# The fold vector is rebuilt rather than loaded, and then checked. A silently different
# fold vector is the one error here that produces a clean-looking wrong answer.
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=42).split(train, y)):
    folds[va] = i
assert (folds >= 0).all() and np.bincount(folds).sum() == len(train)
FOLD_SHA = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
assert FOLD_SHA == "ec282b0968059676", FOLD_SHA
print(f"train {len(train):,}  test {len(test):,}  folds {np.bincount(folds)}")
print(f"fold sha {FOLD_SHA}  VERIFIED")

train 691,369  test 296,302  folds [138274 138274 138274 138274 138273]
fold sha ec282b0968059676  VERIFIED


In [2]:
# Row 77's thirty-five, in row 77's order, then the five candidates.
BASE = [
    ("te42", "te_bag42"), ("te2024", "te_seed2024"), ("te7", "te_seed7"),
    ("te2025", "te_seed2025"), ("te13", "te_seed13"),
    ("anchor", "lgbm_default_anchor_seed42"), ("trees300", "lgbm_trees300_seed42"),
    ("trees1000", "lgbm_trees1000_seed42"), ("trees2000", "lgbm_trees2000_seed42"),
    ("lr010", "lgbm_lr01_n1000_seed42"), ("lr005", "lgbm_lr005_n2000_seed42"),
    ("lr003", "lgbm_lr003_n3333_seed42"),
    ("bag42", "lgbm_bag08_lr005_n2000_seed42"),
    ("bag2024", "lgbm_bag08_lr005_n2000_seed2024"),
    ("bag7", "lgbm_bag08_lr005_n2000_seed7"),
    ("bag2025", "lgbm_bag08_lr005_n2000_seed2025"),
    ("bag13", "lgbm_bag08_lr005_n2000_seed13"),
    ("neural", "neural"), ("cat42", "catboost_te"), ("cat2024", "catboost_te_seed2024"),
    ("cat7", "catboost_te_seed7"), ("cat2025", "catboost_te_seed2025"),
    ("cat13", "catboost_te_seed13"), ("neural_te", "neural_te"),
    ("xgb_te", "xgb_te"), ("xgb2024", "xgb_te_seed2024"), ("xgb7", "xgb_te_seed7"),
    ("xgb2025", "xgb_te_seed2025"), ("xgb13", "xgb_te_seed13"),
    ("pair_top9", "xgb_pair_top9"),
]
BASE += [("cat_nat_c1", "cat_native_c1"), ("cat_nat_c2", "cat_native_c2"),
         ("lgb_raw", "lgb_raw"), ("xgb_raw", "xgb_raw"), ("cat_raw", "cat_raw")]
CAND = [("xgb_te_fe", "xgb_te_fe"), ("cat_te_n4000", "cat_te_n4000"),
        ("xgb_raw_fe", "xgb_raw_fe"), ("cat_raw_n10k", "cat_raw_n10000"),
        ("lgb_raw_fe", "lgb_raw_fe"), ("cat_raw_fe", "cat_raw_fe")]


def load(stem, kind):
    # OOF/test vector. Two naming conventions exist in artifacts/oof. The bare
    # "{stem}.npy" form is the OOF side only: the early LightGBM members never had a
    # test .npy written and their test side lives in submissions/. Falling back to the
    # bare name for kind="test" silently returns the OOF vector, which is caught by the
    # length assert below only because train and test differ in length.
    cands = [O / f"{stem}_{kind}.npy"]
    if kind == "oof":
        cands.append(O / f"{stem}.npy")
    for c in cands:
        if c.exists():
            return np.load(c)
    if kind == "test" and (S / f"{stem}.csv").exists():
        df = pd.read_csv(S / f"{stem}.csv")
        # A csv written in a different row order blends perfectly cleanly and is
        # undetectable in the score. Checked rather than assumed.
        assert (df["id"].to_numpy() == test["id"].to_numpy()).all(), f"id order {stem}"
        return df["addicted_label"].to_numpy()
    raise FileNotFoundError(f"{stem} {kind}")


MEM = BASE + CAND
names = [n for n, _ in MEM]
Poof = {n: load(s, "oof") for n, s in MEM}
Ptest = {n: load(s, "test") for n, s in MEM}

for n in names:
    assert Poof[n].shape == (len(train),), n
    assert Ptest[n].shape == (len(test),), n
    assert np.isfinite(Poof[n]).all() and np.isfinite(Ptest[n]).all(), n
    # A partially failed run leaves a constant fold, which blends silently.
    assert min(np.ptp(Poof[n][folds == f]) for f in range(5)) > 0, f"dead fold in {n}"

# Exact-duplicate quarantine. A duplicated array silently DOUBLES that model's weight.
# Row 59 found xgb_pair_base bit-identical to xgb_te this way and excluded it.
seen = {}
for n in names:
    h = hashlib.md5(np.ascontiguousarray(Poof[n]).tobytes()).hexdigest()
    assert h not in seen, f"{n} is bit-identical to {seen[h]}"
    seen[h] = n
print(f"{len(names)} vectors loaded, no exact duplicates")

41 vectors loaded, no exact duplicates


In [3]:
def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1 - 1e-9)
    return np.clip(np.log(p / (1 - p)), -30, 30)


Loof = np.column_stack([logit(Poof[n]) for n in names])
Ltest = np.column_stack([logit(Ptest[n]) for n in names])
IDX = {n: i for i, n in enumerate(names)}
BASE35 = [IDX[n] for n, _ in BASE]

print("candidate solo CV, and disagreement with the members it most resembles:")
print(f"  {'candidate':12} {'solo CV':>10} {'rho vs xgb_te':>15} {'rho vs cat42':>14}")
for n, _ in CAND:
    cv = np.mean([roc_auc_score(y[folds == f], Poof[n][folds == f]) for f in range(5)])
    r1 = pd.Series(Poof[n]).corr(pd.Series(Poof["xgb_te"]), method="spearman")
    r2 = pd.Series(Poof[n]).corr(pd.Series(Poof["cat42"]), method="spearman")
    print(f"  {n:12} {cv:10.6f} {r1:15.6f} {r2:14.6f}")

# For scale: how decorrelated are two members that everyone agrees are near-copies?
r_seed = pd.Series(Poof["xgb_te"]).corr(pd.Series(Poof["xgb2024"]), method="spearman")
print(f"\n  for scale, xgb_te vs xgb_te_seed2024 (same model, different seed): {r_seed:.6f}")
print("  this repo refuted reasoning from Spearman to blend value on 66 pairs at r=+0.143.")
print("  This table is context, not a prediction.")

candidate solo CV, and disagreement with the members it most resembles:
  candidate       solo CV   rho vs xgb_te   rho vs cat42


  xgb_te_fe      0.968005        0.995305       0.987091


  cat_te_n4000   0.967166        0.991352       0.999208


  xgb_raw_fe     0.964666        0.983258       0.976286


  cat_raw_n10k   0.964042        0.984053       0.983389


  lgb_raw_fe     0.963821        0.978486       0.971446


  cat_raw_fe     0.962181        0.977654       0.980640



  for scale, xgb_te vs xgb_te_seed2024 (same model, different seed): 0.997344
  NOTES.md refuted reasoning from Spearman to blend value on 66 pairs at r=+0.143.
  This table is context, not a prediction.


In [4]:
def run(cols):
    # Fold-wise logistic combiner. No weight is ever fitted on a row it is scored on.
    oof = np.zeros(len(train))
    tst = np.zeros((5, len(test)))
    cf = np.zeros((5, len(cols)))
    nit = []
    for f in range(5):
        tr, va = folds != f, folds == f
        clf = LogisticRegression(C=1.0, max_iter=2000).fit(Loof[np.ix_(tr, cols)], y[tr])
        # A non-converged lbfgs fit reads HIGHER than the truth, so convergence is
        # asserted rather than hoped for. Added 2026-08-21 after the public review
        # flagged it; measured at 38 to 40 iterations, so it has never been close.
        nit.append(int(np.max(clf.n_iter_)))
        oof[va] = clf.decision_function(Loof[np.ix_(va, cols)])
        tst[f] = clf.decision_function(Ltest[:, cols])
        cf[f] = clf.coef_[0]
    assert max(nit) < 2000, f"combiner did not converge, {nit}"
    per = np.array([roc_auc_score(y[folds == f], oof[folds == f]) for f in range(5)])
    return per, tst, cf, max(nit)


ARMS = {"35_row77": BASE35}
for n, _ in CAND:
    ARMS[f"36_{n}"] = BASE35 + [IDX[n]]
ARMS["41_all6"] = BASE35 + [IDX[n] for n, _ in CAND]

res = {a: run(cols) for a, cols in ARMS.items()}
per = {a: r[0] for a, r in res.items()}

ROW77_CV = 0.968110
repro = per["35_row77"].mean() - ROW77_CV
print(f"reproduction of row 77: {per['35_row77'].mean():.6f} vs {ROW77_CV:.6f}"
      f"  delta {repro:+.2e}   {'REPRODUCED' if abs(repro) < 1e-4 else 'FAILED'}")
print(f"combiner max n_iter across all arms: {max(r[3] for r in res.values())} of 2000\n")

hdr = " ".join(f"{'fold ' + str(i):>9}" for i in range(5))
print(f"{'arm':14} {hdr} {'mean':>10} {'sd':>9}")
for a in ARMS:
    print(f"{a:14} " + " ".join(f"{v:9.6f}" for v in per[a])
          + f" {per[a].mean():10.6f} {per[a].std():9.6f}")

reproduction of row 77: 0.968110 vs 0.968110  delta -3.65e-07   REPRODUCED
combiner max n_iter across all arms: 83 of 2000

arm               fold 0    fold 1    fold 2    fold 3    fold 4       mean        sd
35_row77        0.967460  0.968232  0.968365  0.968688  0.967803   0.968110  0.000432
36_xgb_te_fe    0.967902  0.968649  0.968674  0.969159  0.968286   0.968534  0.000420
36_cat_te_n4000  0.967468  0.968243  0.968385  0.968703  0.967811   0.968122  0.000435
36_xgb_raw_fe   0.967986  0.968706  0.968677  0.969159  0.968281   0.968562  0.000400
36_cat_raw_n10k  0.967481  0.968243  0.968383  0.968715  0.967826   0.968130  0.000432
36_lgb_raw_fe   0.967847  0.968583  0.968149  0.969070  0.968231   0.968376  0.000419
36_cat_raw_fe   0.967810  0.968613  0.968671  0.969079  0.968250   0.968484  0.000428
41_all6         0.968107  0.968838  0.968806  0.969332  0.968482   0.968713  0.000407


In [5]:
FLOOR_MEAN, FLOOR_FOLDS = 0.00005, 4
base_per = per["35_row77"]

print("Paired against row 77's thirty-five. The gate is >= +0.00005 mean AND >= 4/5 folds.\n")
print(f"{'arm':14} {'paired mean':>13} {'paired sd':>11} {'folds':>7} {'t(4)':>8}  gate")
gate = {}
for a in ARMS:
    if a == "35_row77":
        continue
    d = per[a] - base_per
    wins = int((d > 0).sum())
    sd = d.std(ddof=1)
    t = d.mean() / (sd / np.sqrt(5)) if sd > 0 else float("inf")
    fired = bool(d.mean() >= FLOOR_MEAN and wins >= FLOOR_FOLDS)
    gate[a] = fired
    print(f"{a:14} {d.mean():+13.6f} {sd:11.6f} {wins:5d}/5 {t:8.2f}"
          f"  {'FIRES' if fired else 'under floor'}")

Paired against row 77's thirty-five. The gate is >= +0.00005 mean AND >= 4/5 folds.

arm              paired mean   paired sd   folds     t(4)  gate
36_xgb_te_fe       +0.000425    0.000069     5/5    13.68  FIRES
36_cat_te_n4000     +0.000012    0.000005     5/5     5.10  under floor
36_xgb_raw_fe      +0.000452    0.000081     5/5    12.43  FIRES
36_cat_raw_n10k     +0.000020    0.000006     5/5     7.65  under floor
36_lgb_raw_fe      +0.000266    0.000271     4/5     2.20  FIRES
36_cat_raw_fe      +0.000375    0.000052     5/5    16.09  FIRES
41_all6            +0.000604    0.000094     5/5    14.31  FIRES


In [6]:
# Coefficients of the best arm, which is where the result actually lives. Row 59's
# lesson: a model that is null on its own can still take a large weight, and where that
# weight comes FROM is the thing worth reading.
best = max((a for a in ARMS if a != "35_row77"), key=lambda a: per[a].mean())
print(f"best arm by CV: {best}   {per[best].mean():.6f}\n")

cols_b, cols_0 = ARMS[best], ARMS["35_row77"]
cb = res[best][2].mean(axis=0)
c0 = res["35_row77"][2].mean(axis=0)
base_map = {names[c]: c0[i] for i, c in enumerate(cols_0)}

rows = []
for i, c in enumerate(cols_b):
    n = names[c]
    was = base_map.get(n, float("nan"))
    rows.append({"member": n, "coef": cb[i], "was": was, "shift": cb[i] - was})
tab = pd.DataFrame(rows).sort_values("coef", ascending=False)
print(tab.to_string(index=False, float_format=lambda v: f"{v:+.4f}"))

new = set(n for n, _ in CAND) & set(tab.member)
gained = tab[tab.member.isin(new)]["coef"].sum()
lost = -tab[~tab.member.isin(new)]["shift"].sum()
print(f"\nnew members carry {gained:+.4f} in total")
print(f"the existing thirty give up {lost:+.4f} of weight between them")
if abs(gained) > 1e-12:
    print(f"substitution covers {100 * lost / gained:.0f} percent of the new weight")

best arm by CV: 41_all6   0.968713

      member    coef     was   shift
  cat_nat_c2 +0.2979 +0.2939 +0.0039
   xgb_te_fe +0.2249     NaN     NaN
  xgb_raw_fe +0.2233     NaN     NaN
  cat_raw_fe +0.1934     NaN     NaN
cat_te_n4000 +0.1833     NaN     NaN
  lgb_raw_fe +0.1436     NaN     NaN
cat_raw_n10k +0.1025     NaN     NaN
   pair_top9 +0.0933 +0.1921 -0.0987
     cat2024 +0.0735 +0.0728 +0.0007
   neural_te +0.0717 +0.1219 -0.0501
     xgb2024 +0.0521 +0.0761 -0.0241
       cat13 +0.0506 +0.0617 -0.0111
       lr003 +0.0426 +0.0905 -0.0479
     cat2025 +0.0417 +0.0575 -0.0157
        xgb7 +0.0344 +0.0567 -0.0223
       lr005 +0.0294 +0.0750 -0.0456
       xgb13 +0.0257 +0.0563 -0.0306
      te2025 +0.0129 +0.0062 +0.0068
   trees1000 +0.0104 +0.0051 +0.0054
        cat7 +0.0081 +0.0438 -0.0357
        te13 +0.0072 +0.0044 +0.0028
      neural +0.0038 +0.0732 -0.0694
     bag2024 +0.0034 +0.0087 -0.0053
     bag2025 +0.0028 +0.0281 -0.0253
     xgb2025 +0.0024 +0.0368 -0.0343
  

In [7]:
# Three decisions, kept separate. Bundling them was the error corrected in row 59.
print("1. GATE")
for a, f in gate.items():
    print(f"     {a:14} {'FIRES' if f else 'under floor'}")

print("\n2. MEMBERSHIP")
print("   A sub-floor addition is still kept and logged as negligible: row 32 kept four")
print("   CatBoost seeds at +0.000014 and row 34 kept neural_te at +0.000043. Membership")
print("   follows the sign and the fold count, not the floor.")
keep = [a for a in ARMS if a != "35_row77"
        and (per[a] - base_per).mean() > 0
        and int(((per[a] - base_per) > 0).sum()) >= 4]
print(f"   arms positive and >= 4/5 folds: {keep if keep else 'none'}")
print(f"   carried forward: {best} at {per[best].mean():.6f}")

print("\n3. SUBMISSION")
SUB = S / "stack_v2.csv"
if per[best].mean() > ROW77_CV:
    p = res[best][1].mean(axis=0)
    sub = pd.DataFrame({"id": test["id"].to_numpy(),
                        "addicted_label": (np.argsort(np.argsort(p)) + 0.5) / len(p)})
    assert len(sub) == len(test) and np.isfinite(sub["addicted_label"]).all()
    sub.to_csv(SUB, index=False)
    print(f"   wrote {SUB.name}, {len(sub):,} rows,"
          f" {sub['addicted_label'].nunique():,} distinct")
    print("   AUC reads order only, so the rank transform changes nothing and keeps the")
    print("   file comparable with the earlier stack submissions.")
else:
    print(f"   no submission: best arm {per[best].mean():.6f} does not beat row 77")

print(f"\nledger lines:\n  name    stack_{best}\n  cv_mean {per[best].mean():.6f}"
      f"\n  cv_std  {per[best].std():.6f}")

1. GATE
     36_xgb_te_fe   FIRES
     36_cat_te_n4000 under floor
     36_xgb_raw_fe  FIRES
     36_cat_raw_n10k under floor
     36_lgb_raw_fe  FIRES
     36_cat_raw_fe  FIRES
     41_all6        FIRES

2. MEMBERSHIP
   A sub-floor addition is still kept and logged as negligible: row 32 kept four
   CatBoost seeds at +0.000014 and row 34 kept neural_te at +0.000043. Membership
   follows the sign and the fold count, not the floor.
   arms positive and >= 4/5 folds: ['36_xgb_te_fe', '36_cat_te_n4000', '36_xgb_raw_fe', '36_cat_raw_n10k', '36_lgb_raw_fe', '36_cat_raw_fe', '41_all6']
   carried forward: 41_all6 at 0.968713

3. SUBMISSION


   wrote stack_v2.csv, 296,302 rows, 296,302 distinct
   AUC reads order only, so the rank transform changes nothing and keeps the
   file comparable with the earlier stack submissions.

ledger lines:
  name    stack_41_all6
  cv_mean 0.968713
  cv_std  0.000407
